# County Rainfall and Planting Advisory

**PLP Data Analytics – Assignment 2**

This notebook cleans the manually entered Machakos rainfall readings, combines monthly rainfall data for five counties, analyses rainfall patterns with NumPy, and exports a county-level summary for an NGO supporting farmers.

**Rainfall units:** millimetres (mm).  
**Month order:** January to December.


## 1. Import Libraries and Define the Dataset

NumPy is used for array operations, calculations, Boolean masks, and identifying the wettest months and planting windows. The built-in `csv` module is used to write the final summary file.


In [1]:
import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = np.array([
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],   # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],  # Mombasa
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

print("Counties:", counties)
print("Number of months:", len(months))

Counties: ['Nairobi', 'Nakuru', 'Kisumu', 'Mombasa', 'Machakos']
Number of months: 12


## 2. Clean Machakos Rainfall Data

The Machakos readings were supplied as text. The cleaning function converts valid values to integers and catches the specific `ValueError` raised when a value cannot be converted. Invalid or missing readings are temporarily represented by `None`; they are then replaced with the rounded mean of the valid readings.


In [2]:
def clean_rainfall(value):
    """Convert a rainfall value to an integer, or return None if invalid."""
    try:
        return int(value.strip())
    except ValueError:
        return None


machakos_clean = [clean_rainfall(value) for value in machakos_raw]

valid_values = [value for value in machakos_clean if value is not None]
machakos_average = round(sum(valid_values) / len(valid_values))

imputed_indices = [index for index, value in enumerate(machakos_clean) if value is None]
imputed_months = months[imputed_indices]

machakos_clean = [
    machakos_average if value is None else value
    for value in machakos_clean
]

print("Valid Machakos readings:", valid_values)
print("Average of valid Machakos readings (rounded):", machakos_average, "mm")
print("Months imputed:", ", ".join(imputed_months))
print("Cleaned Machakos readings:", machakos_clean)

Valid Machakos readings: [48, 35, 88, 140, 20, 9, 12, 45, 182, 96]
Average of valid Machakos readings (rounded): 68 mm
Months imputed: April, September
Cleaned Machakos readings: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 3. Combine County Data into a NumPy Array

The four already-clean county rows are combined with the cleaned Machakos row. The resulting array should contain five rows and twelve columns, with an integer data type.


In [3]:
all_rows = clean_rows + [machakos_clean]
R = np.array(all_rows, dtype=int)

print("Combined rainfall array (mm):")
print(R)
print("Array shape:", R.shape)
print("Array dtype:", R.dtype)

Combined rainfall array (mm):
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Array shape: (5, 12)
Array dtype: int64


## 4. Annual Rainfall Totals and Monthly Averages

The annual total is calculated across columns for each county (`axis=1`). The monthly average is calculated down the county rows for each month (`axis=0`).


In [4]:
annual_totals = np.sum(R, axis=1)
monthly_averages = np.mean(R, axis=0)

print("Annual rainfall totals by county (mm):")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\\nAverage rainfall across the five counties by month (mm):")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")

Annual rainfall totals by county (mm):
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm
\nAverage rainfall across the five counties by month (mm):
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


## 5. Wettest Month for Each County

`np.argmax()` with `axis=1` returns the column index of the highest monthly rainfall in each county. The indices are used to retrieve month names, and `zip()` pairs each county with its wettest month.


In [5]:
wettest_indices = np.argmax(R, axis=1)
wettest_months = months[wettest_indices]
wettest_rainfall = np.max(R, axis=1)

print("Wettest month by county:")
for county, month, rainfall in zip(counties, wettest_months, wettest_rainfall):
    print(f"{county}: {month} ({rainfall} mm)")

Wettest month by county:
Nairobi: April (219 mm)
Nakuru: April (145 mm)
Kisumu: April (240 mm)
Mombasa: May (240 mm)
Machakos: November (182 mm)


## 6. Count Dry Months

A dry month is defined in this assignment as a month with rainfall strictly below 50 mm. A Boolean mask identifies those readings, and `np.sum()` counts the `True` values for each county.


In [6]:
dry_mask = R < 50
dry_month_counts = np.sum(dry_mask, axis=1)

print("Number of dry months (below 50 mm) by county:")
for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count}")

Number of dry months (below 50 mm) by county:
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


## 7. Identify the Recommended Planting Window

For each county, adjacent monthly rainfall values are added using the shifted-array technique. `np.argmax()` identifies the pair of consecutive months with the highest combined rainfall. This is a rainfall-based recommendation only; it does not account for soil moisture, crop type, forecast conditions, or farm-level drainage.


In [7]:
consecutive_totals = R[:, :-1] + R[:, 1:]
planting_start_indices = np.argmax(consecutive_totals, axis=1)
planting_windows = np.array([
    f"{months[index]}–{months[index + 1]}"
    for index in planting_start_indices
])
planting_window_rainfall = np.max(consecutive_totals, axis=1)

print("Recommended two-month planting windows:")
for county, window, total in zip(counties, planting_windows, planting_window_rainfall):
    print(f"{county}: {window} ({total} mm combined)")

Recommended two-month planting windows:
Nairobi: April–May (395 mm combined)
Nakuru: April–May (265 mm combined)
Kisumu: April–May (420 mm combined)
Mombasa: April–May (390 mm combined)
Machakos: November–December (278 mm combined)


## Planting Advisory

The rainfall analysis suggests that the NGO should prioritise **April–May** for Nairobi, Nakuru, Kisumu, and Mombasa, while **November–December** is the strongest two-month rainfall window for Machakos. These windows identify the consecutive months with the highest combined historical rainfall in the supplied dataset, so they can guide the timing of county-level SMS messages. Farmers should still consider crop type, soil condition, local drainage, and updated weather forecasts before planting. The NGO should advise farmers to prepare land and obtain suitable seed ahead of the identified windows rather than waiting for the rains to begin. **Data-quality caveat:** Machakos had two invalid or missing entries, for April and September, which were imputed using the rounded mean of its ten valid readings (68 mm); this may affect its totals and recommended window. The dataset contains only one set of monthly readings and should not be treated as a long-term climate normal or a guarantee of future rainfall.


## Data Quality Notes

- Machakos was the only county supplied as raw text and required cleaning.
- `"N/A"` and the empty September entry could not be converted to integers and were replaced with the rounded mean of the ten valid readings: **68 mm**.
- The June value `" 20 "` contained surrounding spaces; stripping whitespace allowed it to be converted successfully to `20`.
- Imputation assumes the valid Machakos observations provide a reasonable estimate for the two missing months. This is a simple assignment method, not a substitute for checking the original weather-station records.
- All rainfall values are treated as monthly totals in millimetres. The dataset does not provide multiple years, station metadata, or crop-specific requirements.


## 8. Export the Rainfall Summary CSV

The CSV contains exactly one row per county and the required five columns. The built-in `csv` module writes the file in the repository's working directory.


In [8]:
summary_headers = [
    "county",
    "annual_total_mm",
    "wettest_month",
    "dry_months",
    "planting_window"
]

summary_rows = zip(
    counties,
    annual_totals,
    wettest_months,
    dry_month_counts,
    planting_windows
)

with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(summary_headers)
    writer.writerows(summary_rows)

print("Created rainfall_summary.csv with", len(counties), "county rows.")
print("CSV columns:", summary_headers)

Created rainfall_summary.csv with 5 county rows.
CSV columns: ['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']


## 9. Conclusion

NumPy was used to calculate annual totals, monthly averages, wettest months, dry-month counts, and consecutive-month rainfall windows. The cleaned results were exported to `rainfall_summary.csv` for use by the NGO. The advisory should be used alongside local knowledge and current forecasts, especially because two Machakos readings were estimated rather than observed.
